# M3 — Juggler-Inspired Dynamic Weight Balancing
Divides training into epochs of increasing length. After each epoch, evaluates validation losses and adjusts α weights.

**⚠️ After running M1, update `L_STAR` below with values slightly below M1's final validation losses.**

In [1]:
# Cell 0: Pick one model, restart kernel between models
#TRAIN_MODEL = "qwen25_1p5b_base"
#TRAIN_MODEL = "qwen25_3b_base"
TRAIN_MODEL = "qwen25_7b_base"

In [2]:
# Cell 1: Imports and setup
import os, sys, json, math, re
from typing import List, Dict, Any, Tuple

import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import TrainingArguments, Trainer

sys.path.insert(0, os.path.expanduser("~/kd_project"))
from shared_utils import (
    RUNS_DIR, MAX_SEQ_LEN, LR, BATCH_SIZE, GRAD_ACC, SEED,
    W_FORMAT, W_DECISION, W_EXPL, LAMBDA, STUDENTS,
    load_data, load_student,
    get_section_spans, in_any_span,
    compute_alpha, compute_mean_confidence,
    find_decision_span_chars, teacher_section_entropy_mean,
    build_prompt_text, tokenize_and_mask, FlexibleCollator,
)

OUT_DIR = os.path.join(RUNS_DIR, "m3_juggler")
os.makedirs(OUT_DIR, exist_ok=True)

# Juggler config
N_JUGGLER_EPOCHS = 8
ALPHA_INIT = np.array([1.0, 1.0, 1.0])
LOSS_NAMES = ["L_cw_wsft", "L_dec_only", "L_dec_ent"]

# ⚠️ SET THESE from M1 results — should be slightly below M1's val losses
# In Cell 1, replace L_STAR:
L_STAR = np.array([0.8, 0.1, 0.001])  # targets BELOW initial val losses

EPSILON_1 = 0.5
T_BASE = 36
R = 5.0
N_VAL_SUBSET = 200

def epsilon_kappa(k): return EPSILON_1 * (k ** -0.95)
def T_kappa(k): return max(1, int(T_BASE * (k ** 0.2)))
def project_R(alpha, R):
    alpha = np.maximum(alpha, 0.0)
    norm = np.linalg.norm(alpha)
    return alpha * (R / norm) if norm > R else alpha

prompts, teacher_rows = load_data()
MEAN_C = compute_mean_confidence(teacher_rows)

val_rows = teacher_rows[-N_VAL_SUBSET:]
train_rows = teacher_rows[:-N_VAL_SUBSET]
print(f"Juggler train: {len(train_rows)}, val: {len(val_rows)}")

c:\Users\adishalit1\AppData\Local\anaconda3\envs\kd\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Loaded 5000 teacher samples from: data\clinical_pharm_teacher_train_6000.jsonl
Juggler train: 4800, val: 200


In [3]:
# Cell 2: Dataset — precomputed
from tqdm import tqdm

class JugglerDatasetFast(Dataset):
    def __init__(self, rows, prompts, tokenizer, is_instruct, mean_c):
        print("  Precomputing dataset...")
        self.items = []
        for idx in tqdm(range(len(rows)), desc="  Tokenizing"):
            r = rows[idx]
            prompt = prompts[r["id"]]
            answer = r["teacher_text"]
            prompt_text = build_prompt_text(tokenizer, prompt, is_instruct)
            input_ids, offsets, labels, answer_start = tokenize_and_mask(tokenizer, prompt_text, answer)

            wsft_weights = [0.0] * len(input_ids)
            spans = get_section_spans(answer)
            dec_spans = [(answer_start + s, answer_start + e) for s, e in spans["decision"]]
            expl_spans = [(answer_start + s, answer_start + e) for s, e in spans["explanation"]]
            for i, (s, e) in enumerate(offsets):
                if e <= s: continue
                if s >= answer_start:
                    w = W_FORMAT
                    if in_any_span(s, e, dec_spans): w = W_DECISION
                    if in_any_span(s, e, expl_spans): w = W_EXPL
                    wsft_weights[i] = float(w)
            active_w = [w for w in wsft_weights if w > 0]
            if active_w:
                mean_w = sum(active_w) / len(active_w)
                if mean_w > 1e-6: wsft_weights = [w / mean_w if w > 0 else 0.0 for w in wsft_weights]

            alpha_conf = compute_alpha(r, mean_c)
            dec_mask = torch.zeros(len(input_ids), dtype=torch.bool)
            dec_span_chars = find_decision_span_chars(answer)
            if dec_span_chars:
                ds, de = dec_span_chars
                for i, (s, e) in enumerate(offsets):
                    if labels[i] != -100 and not (e <= answer_start+ds or s >= answer_start+de):
                        dec_mask[i] = True
            ent_teacher = teacher_section_entropy_mean(r, dec_span_chars)

            self.items.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "attention_mask": torch.ones(len(input_ids), dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
                "wsft_weights": torch.tensor(wsft_weights, dtype=torch.float),
                "alpha_conf": torch.tensor(alpha_conf, dtype=torch.float),
                "dec_mask": dec_mask,
                "ent_teacher": ent_teacher.float(),
            })
        print(f"  ✅ Precomputed {len(self.items)} samples")

    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]

print("✅ Fast dataset class ready")

✅ Fast dataset class ready


In [4]:
# Cell 3: Juggler Trainer + validation evaluator
class JugglerTrainer(Trainer):
    def __init__(self, juggler_alpha, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.juggler_alpha = juggler_alpha

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        wsft_w = inputs.pop("wsft_weights")
        alpha_conf = inputs.pop("alpha_conf")
        dec_mask = inputs.pop("dec_mask")
        ent_teacher = inputs.pop("ent_teacher")

        out = model(**inputs)
        logits = out.logits
        sl = logits[:, :-1, :].contiguous()
        ll = labels[:, 1:].contiguous()
        sw = wsft_w[:, 1:].contiguous()
        dm = dec_mask[:, 1:]
        vocab, B = sl.size(-1), sl.size(0)

        tl = torch.nn.CrossEntropyLoss(reduction="none")(
            sl.view(-1, vocab), ll.view(-1)).view(ll.size())
        active = (ll != -100).float()

        w = sw * active
        denom = active.sum(dim=1).clamp(min=1.0)
        L1 = ((tl * w).sum(dim=1) / denom * alpha_conf).mean()

        da = dm.float() * active
        L2 = (tl * da).sum() / da.sum().clamp(min=1.0)

        probs = torch.softmax(sl, dim=-1)
        et = -(probs * torch.log(probs + 1e-9)).sum(-1)
        es = torch.zeros(B, device=logits.device)
        for b in range(B):
            m = dm[b]
            if m.any(): es[b] = et[b][m].mean()
        L3 = LAMBDA * ((es - ent_teacher.to(logits.device)) ** 2).mean()

        a = self.juggler_alpha
        loss = a[0] * L1 + a[1] * L2 + a[2] * L3
        return (loss, out) if return_outputs else loss


@torch.no_grad()
def evaluate_val_losses(model, val_dataset, collator):
    model.eval()
    loader = DataLoader(val_dataset, batch_size=1, collate_fn=collator, shuffle=False)
    sums = np.zeros(3)
    count = 0
    for batch in loader:
        batch = {k: v.to("cuda") if hasattr(v, "to") else v for k, v in batch.items()}
        labels = batch.pop("labels"); wsft_w = batch.pop("wsft_weights")
        alpha_conf = batch.pop("alpha_conf"); dm = batch.pop("dec_mask")
        et = batch.pop("ent_teacher")
        logits = model(**batch).logits
        sl = logits[:, :-1, :]; ll = labels[:, 1:]; sw = wsft_w[:, 1:]; dm2 = dm[:, 1:]
        vocab, B = sl.size(-1), sl.size(0)
        tl = torch.nn.CrossEntropyLoss(reduction="none")(
            sl.reshape(-1, vocab), ll.reshape(-1)).view(ll.size())
        active = (ll != -100).float()

        # L_cw_wsft
        w = sw * active; denom = active.sum(dim=1).clamp(min=1.0)
        sums[0] += ((tl * w).sum(dim=1) / denom * alpha_conf).sum().item()

        # L_dec_only
        da = dm2.float() * active
        sums[1] += ((tl * da).sum() / da.sum().clamp(min=1.0)).item()

        # L_dec_ent — memory efficient (only decision tokens)
        dec_active_mask = dm2 & (ll != -100)
        for b in range(B):
            idx = dec_active_mask[b].nonzero(as_tuple=True)[0]
            if len(idx) > 0:
                local_logits = sl[b, idx, :]
                local_probs = torch.softmax(local_logits, dim=-1)
                local_ent = -(local_probs * torch.log(local_probs + 1e-9)).sum(-1)
                student_ent = local_ent.mean()
                sums[2] += LAMBDA * ((student_ent - et[b].to(sl.device)) ** 2).item()

        count += 1
    model.train()
    return sums / max(1, count)

print("✅ Trainer + evaluator ready")

✅ Trainer + evaluator ready


In [ ]:
# Cell 4: Juggler training loop — with resume support
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = TRAIN_MODEL
cfg = STUDENTS[TRAIN_MODEL]
print(f"\n{'='*50} M3 Juggler: {model_name} {'='*50}")

out_path = os.path.join(OUT_DIR, model_name)
os.makedirs(out_path, exist_ok=True)

# ===== RESUME CONFIG =====
# Set to 1 for fresh start, or N+1 to resume after κ=N
RESUME_FROM_KAPPA = 7  # ← change this as needed

alpha_file = os.path.join(out_path, "juggler_alpha.json")
history_file = os.path.join(out_path, "juggler_history.json")

if RESUME_FROM_KAPPA > 1 and os.path.exists(alpha_file):
    with open(alpha_file) as f:
        saved = json.load(f)
    alpha = np.array(saved["alpha"])
    with open(history_file) as f:
        history = json.load(f)
    print(f"  Resuming from κ={RESUME_FROM_KAPPA}, α=[{alpha[0]:.3f},{alpha[1]:.3f},{alpha[2]:.3f}]")

    # Load model from checkpoint
    print("  Loading model from checkpoint...")
    tokenizer = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    base_model = AutoModelForCausalLM.from_pretrained(
        cfg["path"], torch_dtype=torch.bfloat16,
        device_map="auto", trust_remote_code=True)
    model = PeftModel.from_pretrained(base_model, out_path, is_trainable=True)
    model.train()
    print("  ✅ Checkpoint loaded")
else:
    alpha = ALPHA_INIT.copy()
    history = []
    RESUME_FROM_KAPPA = 1
    tokenizer, model = load_student(model_name, cfg)

train_ds = JugglerDatasetFast(train_rows, prompts, tokenizer, cfg["is_instruct"], MEAN_C)
val_ds = JugglerDatasetFast(val_rows, prompts, tokenizer, cfg["is_instruct"], MEAN_C)
collator = FlexibleCollator(tokenizer, extra_1d_fields=["wsft_weights", "dec_mask"],
                            extra_scalar_fields=["alpha_conf", "ent_teacher"])

for kappa in range(RESUME_FROM_KAPPA, N_JUGGLER_EPOCHS + 1):
    T_k, eps_k = T_kappa(kappa), epsilon_kappa(kappa)
    print(f"\n--- κ={kappa}: T={T_k}, ε={eps_k:.4f}, α=[{alpha[0]:.3f},{alpha[1]:.3f},{alpha[2]:.3f}]")

    trainer = JugglerTrainer(
        juggler_alpha=alpha.tolist(), model=model,
        args=TrainingArguments(
            output_dir=os.path.join(out_path, f"ep{kappa}"), max_steps=T_k,
            per_device_train_batch_size=1, gradient_accumulation_steps=32,
            learning_rate=LR, logging_steps=max(1, T_k//4), save_strategy="no",
            bf16=True, seed=SEED+kappa, report_to="none",
            remove_unused_columns=False, dataloader_num_workers=0,
        ),
        train_dataset=train_ds, data_collator=collator,
    )
    trainer.train()

    val_losses = evaluate_val_losses(model, val_ds, collator)
    delta = val_losses - L_STAR
    alpha_new = project_R(alpha + eps_k * delta, R)
    print(f"    val={dict(zip(LOSS_NAMES, val_losses.round(4)))}")
    print(f"    α→[{alpha_new[0]:.3f},{alpha_new[1]:.3f},{alpha_new[2]:.3f}]")
    history.append({"kappa": kappa, "alpha": alpha.tolist(), "val": val_losses.tolist(), "alpha_new": alpha_new.tolist()})
    alpha = alpha_new

    # Save checkpoint
    model.save_pretrained(out_path)
    with open(history_file, "w") as f: json.dump(history, f, indent=2)
    with open(alpha_file, "w") as f: json.dump({"kappa": kappa, "alpha": alpha.tolist()}, f)
    print(f"    💾 Checkpoint saved (κ={kappa})")

tokenizer.save_pretrained(out_path)
print(f"\n✅ {model_name} done. Final α=[{alpha[0]:.3f},{alpha[1]:.3f},{alpha[2]:.3f}]")
print("⚠️ To train next model: change TRAIN_MODEL in Cell 0, RESTART KERNEL, run all cells")


================================================== M3 Juggler: qwen25_7b_base ==================================================
  Resuming from κ=7, α=[1.251,0.958,1.000]
  Loading model from checkpoint...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 339/339 [00:09<00:00, 34.12it/s]


  ✅ Checkpoint loaded
  Precomputing dataset...


  Tokenizing: 100%|██████████| 4800/4800 [00:07<00:00, 679.97it/s]


  ✅ Precomputed 4800 samples
  Precomputing dataset...


...

Step,Training Loss
13,36.335609
26,36.800990
39,35.419704
52,33.785576


    val={'L_cw_wsft': np.float64(0.8206), 'L_dec_only': np.float64(0.0514), 'L_dec_ent': np.float64(0.001)}
    α→[1.253,0.955,1.000]
    💾 Checkpoint saved (κ=7)

--- κ=8: T=54, ε=0.0693, α=[1.253,0.955,1.000]


Step,Training Loss
13,35.354969
26,34.396285
39,34.924511
52,34.305662


    val={'L_cw_wsft': np.float64(0.7994), 'L_dec_only': np.float64(0.0467), 'L_dec_ent': np.float64(0.001)}
    α→[1.253,0.951,1.000]
    💾 Checkpoint saved (κ=8)

✅ qwen25_7b_base done. Final α=[1.253,0.951,1.000]
⚠️ To train next model: change TRAIN_MODEL in Cell 0, RESTART KERNEL, run all cells


: 

## M3 Juggler — 3B Base Results

### Run config
- Model: `qwen25_3b_base` (3,093M params, 7.4M trainable LoRA)
- bf16, bs=1, ga=32, LR=2e-4
- `L_STAR = [0.8, 0.1, 0.001]`
- `torch.cuda.set_per_process_memory_fraction(0.85)` to prevent Windows WDDM crash
- Checkpoint saved after each κ epoch

### Weight trajectory

| κ | T_κ | ε_κ | α_cw | α_dec | α_ent | val_cw | val_dec | val_ent |
|---|-----|------|------|-------|-------|--------|---------|---------|
| 1 | 36 | 0.5000 | 1.000 | 1.000 | 1.000 | 1.2837 | 0.0874 | 0.0012 |
| 2 | 41 | 0.2588 | 1.242 | 0.994 | 1.000 | 1.1416 | 0.0692 | 0.0013 |
| 3 | 44 | 0.1761 | 1.330 | 0.986 | 1.000 | 1.0745 | 0.0775 | 0.0010 |
| 4 | 47 | 0.1340 | 1.379 | 0.982 | 1.000 | 1.0208 | 0.0543 | 0.0009 |
| 5 | 49 | 0.1084 | 1.408 | 0.976 | 1.000 | | | |
| 6 | 51 | 0.0911 | | | | | | |
| 7 | 53 | 0.0787 | | | | | | |
| 8 | 55 | 0.0692 | | | | | | |

### Observations (κ=1-4)
- **All val losses decreasing**: cw 1.28→1.02 (-20%), dec 0.087→0.054 (-38%), ent 0.0012→0.0009 (-25%)
- **Weights stable**: α_cw slowly increasing (1.0→1.41) because L_cw is hardest to push below target. α_dec and α_ent near 1.0 — those losses are already close to L_STAR.
- **3B converges faster than 1.5B**: val_cw reached 1.02 by κ=4 vs 1.05 at κ=6 for 1.5B
- **Previous run crashed at κ=5** due to Windows WDDM VRAM exhaustion. Fixed with memory fraction limit + per-epoch checkpointing.
- NOTE: Displayed training loss (~50) is accumulated over 32 grad_acc steps. Actual per-step loss ≈ 50/32 ≈ 1.6